# 01 — Setup & data shards (Colab T4)

Clone **midi-llm** + **musicinstruct**, install dependencies, authenticate Hugging Face, and build shards for **Paper 2** training stages:

- **S3** — MIDI-Instruct edit shards (REMI, AMT, Octuple; ABC disabled)
- **S2** *(optional)* — [MidiCaps](https://huggingface.co/datasets/amaai-lab/MidiCaps) caption↔MIDI shards
- **S0** *(optional)* — GigaMIDI syntax shard

**Each notebook uses its own Colab runtime** (you cannot share a VM across notebook tabs). Set **`DATASET_MODE`** to `pilot` (synthetic, 300 items) or `real_v0.2` (MidiCaps/Lakh, ~955 items). Dataset files are reused from **`MyDrive/musicinstruct/data/`** when available — no duplicate copy to `midi-llm/pilot/`. Run **Sync to Drive** at the end so notebooks **02** and **03** restore shards in a new session.

**Runtime:** GPU → T4 (or better).

In [ ]:
# @title Configuration — edit if your forks differ
MIDI_LLM_REPO = "https://github.com/juliagsy/midi-llm.git"
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # or your feature branch

WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
MUSICINSTRUCT = f"{WORK}/musicinstruct"

DATASET_MODE = "pilot"  # @param ["pilot", "real_v0.2"]  synthetic vs MidiCaps/Lakh
RUN_ID = "real_v0.2_v1" if DATASET_MODE == "real_v0.2" else "pilot_v1"  # reuse in 02/03
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"
SHARDS_ROOT = f"{WORK}/data/shards/{RUN_ID}"

USE_DRIVE = True  # @param {type:"boolean"}  recommended for multi-notebook workflow
DRIVE_MIDI_LLM = "/content/drive/MyDrive/midi-llm"
DRIVE_MUSICINSTRUCT = "/content/drive/MyDrive/musicinstruct"
DRIVE_SHARDS = f"{DRIVE_MIDI_LLM}/shards/{RUN_ID}"
DRIVE_MIDICAPS_TAR = f"{DRIVE_MIDI_LLM}/midicaps/midicaps.tar.gz"
# Reuse musicinstruct tarball: DRIVE_MIDICAPS_TAR = f"{DRIVE_MUSICINSTRUCT}/midicaps/midicaps.tar.gz"

USE_MUSICINSTRUCT_DRIVE = True  # @param {type:"boolean"}  symlink dataset from musicinstruct Drive
TARGET_ITEMS = 300 if DATASET_MODE == "pilot" else 1000  # @param {type:"integer"}
FORCE_REGENERATE = False  # @param {type:"boolean"}  overwrite local data before generate

# Real v0.2 only — MidiCaps tarball, extract locally each session
MIDICAPS_ROOT = f"{WORK}/data/midicaps"
LAKH_ROOT = f"{MIDICAPS_ROOT}/lmd_full"
SEED_LIMIT = 200  # @param {type:"integer"}
MIDICAPS_SCAN_LIMIT = 5000  # @param {type:"integer"}

In [ ]:
# @title Mount Google Drive (optional)
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    Path(DRIVE_SHARDS).parent.mkdir(parents=True, exist_ok=True)
    print("Drive shards:", DRIVE_SHARDS)
    print("Drive musicinstruct data:", f"{DRIVE_MUSICINSTRUCT}/data")
else:
    print("Drive disabled — artifacts stay on this VM only.")

In [ ]:
# @title Hugging Face token (Llama 3.2 license required)
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# @title Clone repos
!rm -rf {MIDI_LLM} {MUSICINSTRUCT}
!git clone --branch {BRANCH} --depth 1 {MIDI_LLM_REPO} {MIDI_LLM}
!git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}
%cd {MIDI_LLM}

In [ ]:
# @title Install midi-llm + musicinstruct (T4-friendly stack)
# Colab torchao breaks peft; LoRA does not need it
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33"
!pip install -q -e ".[repr,data,train]"
!pip install -q -e "{MUSICINSTRUCT}"
# Optional AMT arm (GitHub install)
!pip install -q "git+https://github.com/jthickstun/anticipation.git" mido

In [ ]:
# @title GPU check
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title Restore or generate MIDI-Instruct dataset
import shutil
from pathlib import Path

from midi_llm.instruct_dataset import ensure_instruct_data_local, resolve_instruct_paths

paths = resolve_instruct_paths(
    mode=DATASET_MODE,
    musicinstruct_root=MUSICINSTRUCT,
    run_id=RUN_ID,
    drive_musicinstruct_root=DRIVE_MUSICINSTRUCT if USE_DRIVE else None,
)
MANIFEST = str(paths.manifest_path)
LOCAL_DATA = str(paths.data_dir)
MANIFEST_NAME = paths.manifest_name

if FORCE_REGENERATE and paths.data_dir.exists():
    if paths.data_dir.is_symlink():
        paths.data_dir.unlink()
    else:
        shutil.rmtree(paths.data_dir)

manifest = paths.manifest_path
if not manifest.is_file() and USE_DRIVE and USE_MUSICINSTRUCT_DRIVE:
    try:
        ensure_instruct_data_local(paths, use_drive=True, prefer_symlink=True)
        print("Restored from musicinstruct Drive:", paths.drive_data_dir)
    except FileNotFoundError as exc:
        print(exc)

if not manifest.is_file():
    if DATASET_MODE == "pilot":
        force_flag = "--force" if paths.data_dir.exists() else ""
        !cd {MUSICINSTRUCT} && musicinstruct generate-pilot --output-dir {LOCAL_DATA} --target {TARGET_ITEMS} {force_flag}
    else:
        if not Path(LAKH_ROOT).is_dir():
            raise FileNotFoundError(
                f"No Lakh tree at {LAKH_ROOT}. Run the MidiCaps cell below with BUILD_MIDICAPS=True first, "
                "or run musicinstruct colab 01."
            )
        force_flag = "--force" if paths.data_dir.exists() else ""
        !cd {MUSICINSTRUCT} && musicinstruct generate-real --lakh-root {LAKH_ROOT} --output-dir {LOCAL_DATA} \
            --target {TARGET_ITEMS} --seed-limit {SEED_LIMIT} \
            --midicaps-limit {MIDICAPS_SCAN_LIMIT} --manifest-name {MANIFEST_NAME} {force_flag}
else:
    print("Using manifest:", manifest)

!cd {MUSICINSTRUCT} && musicinstruct validate {MANIFEST}
!cd {MUSICINSTRUCT} && musicinstruct self-test {MANIFEST}

In [ ]:
# @title Build S3 edit shards (train + validation) for all representations
import os
from pathlib import Path

os.makedirs(SHARDS_ROOT, exist_ok=True)
manifest = MANIFEST
reprs = ["remi", "octuple", "amt"]

for split in ("train", "validation"):
    for repr_name in reprs:
        out = f"{SHARDS_ROOT}/edit_{repr_name}_{split}.jsonl"
        !midi-llm build-instruct-shard {manifest} --repr {repr_name} --output {out} --split {split}

print("Shards written to:", SHARDS_ROOT)
!ls -la {SHARDS_ROOT}

In [ ]:
# @title Optional — MidiCaps caption↔MIDI shards (S2)
BUILD_MIDICAPS = False  # @param {type:"boolean"}
MIDICAPS_LIMIT = 500  # @param {type:"integer"}  cap examples per repr (×2 if bidirectional)

import os
import shutil
import subprocess
import tarfile
from pathlib import Path

# Do NOT use hf_hub_download here — Colab's preloaded huggingface_hub/xet often breaks
# on this 1.6 GB XET-backed file even after pip upgrade (hub already imported above).


MIDICAPS_HF_URL = "https://huggingface.co/datasets/amaai-lab/MidiCaps/resolve/main/midicaps.tar.gz"
MIDICAPS_MIN_BYTES = 1_000_000_000


def _valid_midicaps_tar(path: Path) -> bool:
    return path.is_file() and path.stat().st_size > MIDICAPS_MIN_BYTES


def _download_midicaps_tar(dest: Path) -> Path:
    """Download midicaps.tar.gz via HTTP (wget -L), bypassing huggingface_hub XET."""
    if _valid_midicaps_tar(dest):
        print("Using cached", dest)
        return dest

    dest.parent.mkdir(parents=True, exist_ok=True)
    token = os.environ.get("HF_TOKEN")
    cmd = ["wget", "-c", "-L", "--show-progress", "-O", str(dest), MIDICAPS_HF_URL]
    if token:
        cmd = [
            "wget",
            "-c",
            "-L",
            "--show-progress",
            "--header",
            f"Authorization: Bearer {token}",
            "-O",
            str(dest),
            MIDICAPS_HF_URL,
        ]
    print("Downloading via wget (not hf_hub_download)...")
    subprocess.run(cmd, check=True)
    size = dest.stat().st_size
    if size < MIDICAPS_MIN_BYTES:
        dest.unlink(missing_ok=True)
        raise RuntimeError(f"Download too small ({size} bytes) — retry or check network")
    return dest


def _extract_midicaps_tar(tar_path: Path, midicaps_root: Path) -> None:
    marker = midicaps_root / "lmd_full"
    if marker.is_dir():
        print("Using existing extract at", marker)
        return
    midicaps_root.mkdir(parents=True, exist_ok=True)
    print("Extracting", tar_path, "→", midicaps_root, "(~5–15 min on VM disk)")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=midicaps_root)


def ensure_midicaps_local() -> Path:
    """Restore tarball from Drive if needed, then extract locally."""
    midicaps_root = Path(MIDICAPS_ROOT)
    local_tar = midicaps_root / "midicaps.tar.gz"
    drive_tar = Path(DRIVE_MIDICAPS_TAR)

    if (midicaps_root / "lmd_full").is_dir():
        print("Using existing MidiCaps at", midicaps_root)
        return local_tar if _valid_midicaps_tar(local_tar) else drive_tar

    if USE_DRIVE and _valid_midicaps_tar(drive_tar):
        midicaps_root.mkdir(parents=True, exist_ok=True)
        if not _valid_midicaps_tar(local_tar):
            print("Copying tarball from Drive (one file, ~1.6 GB)...")
            shutil.copy2(drive_tar, local_tar)
        _extract_midicaps_tar(local_tar, midicaps_root)
        return local_tar

    if not _valid_midicaps_tar(local_tar):
        print("Downloading midicaps.tar.gz (~1.6 GB from Hugging Face)...")
        _download_midicaps_tar(local_tar)
    _extract_midicaps_tar(local_tar, midicaps_root)
    return local_tar


if BUILD_MIDICAPS:
    os.makedirs(SHARDS_ROOT, exist_ok=True)
    ensure_midicaps_local()

    for repr_name in ("remi", "octuple", "amt"):
        out = f"{SHARDS_ROOT}/midicaps_{repr_name}_train.jsonl"
        !midi-llm build-midicaps-shard --midi-root {MIDICAPS_ROOT} --repr {repr_name} --output {out} --limit {MIDICAPS_LIMIT}

    print("MidiCaps shards:")
    !ls -la {SHARDS_ROOT}/midicaps_*
else:
    print("Skipped MidiCaps (set BUILD_MIDICAPS=True to enable S2 shards)")

In [ ]:
# @title Validate shards (line counts + Octuple wire format)
import json
from pathlib import Path

reprs = ["remi", "octuple", "amt"]
manifest = Path(MANIFEST)
sample_midi = None
with manifest.open(encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        if row.get("split") == "train":
            candidate = manifest.parent / row["midi_in"]
            if candidate.is_file():
                sample_midi = candidate
                break

for repr_name in reprs:
    shard = Path(SHARDS_ROOT) / f"edit_{repr_name}_train.jsonl"
    lines = shard.read_text(encoding="utf-8").strip().splitlines()
    record = json.loads(lines[0])
    completion = record["completion"]
    assert record["repr_name"] == repr_name, (repr_name, record.get("repr_name"))
    print(f"{repr_name}: {len(lines)} train examples, repr_name={record['repr_name']}")
    if repr_name == "octuple":
        assert "|" in completion and "[" not in completion, "bad octuple payload (expected a,b,c|d,e,f)"
    else:
        assert "[" not in completion, f"bad {repr_name} payload (expected flat integers)"

if sample_midi:
    !midi-llm roundtrip {sample_midi} --repr octuple --json
    !midi-llm token-stats {sample_midi} --repr octuple --no-bpe
else:
    print("Skipped roundtrip smoke (no train midi_in found)")

In [ ]:
# @title Sync shards to Google Drive (for notebooks 02/03 in a new runtime)
import shutil
from pathlib import Path

if USE_DRIVE:
    Path(DRIVE_SHARDS).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SHARDS_ROOT, DRIVE_SHARDS, dirs_exist_ok=True)
    print("Synced shards →", DRIVE_SHARDS)
    print("Files:", sorted(p.name for p in Path(DRIVE_SHARDS).glob("*.jsonl")))
    print("Dataset (manifest + midi):", MANIFEST)
    if USE_MUSICINSTRUCT_DRIVE:
        print("Dataset reused from musicinstruct Drive — not duplicated under midi-llm/")
    else:
        drive_data = Path(DRIVE_MUSICINSTRUCT) / paths.data_dir.relative_to(Path(MUSICINSTRUCT))
        drive_data.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(paths.data_dir, drive_data, dirs_exist_ok=True)
        print("Synced dataset →", drive_data)
    local_tar = Path(MIDICAPS_ROOT) / "midicaps.tar.gz"
    drive_tar = Path(DRIVE_MIDICAPS_TAR)
    if local_tar.is_file() and local_tar.stat().st_size > 1_000_000_000:
        drive_tar.parent.mkdir(parents=True, exist_ok=True)
        if drive_tar.is_file() and drive_tar.stat().st_size >= local_tar.stat().st_size:
            print("Drive tarball already up to date:", drive_tar)
        else:
            print("Syncing MidiCaps tarball → Drive (single file, ~10–25 min)...")
            shutil.copy2(local_tar, drive_tar)
            print("Synced →", drive_tar)
else:
    print("Skipped Drive sync. Shards are only on this VM:", SHARDS_ROOT)
    print("Re-run notebook 01 in notebook 02/03, or enable USE_DRIVE here.")

In [ ]:
# @title Optional — GigaMIDI syntax shard (S0, no Lakh)
BUILD_GIGAMIDI = False  # @param {type:"boolean"}
GIGAMIDI_LIMIT = 200  # @param {type:"integer"}

if BUILD_GIGAMIDI:
    out = f"{SHARDS_ROOT}/syntax_remi_gigamidi.jsonl"
    !midi-llm build-gigamidi-shard --repr remi --output {out} --limit {GIGAMIDI_LIMIT}
else:
    print("Skipped GigaMIDI (set BUILD_GIGAMIDI=True to enable)")

In [ ]:
# @title Training time estimate on this GPU
import json

from midi_llm.train.estimate import estimate_training

_seq_len = 1024  # match notebook 02 T4 override; protocol default is load_config("remi")["model"]["max_seq_len"]
for steps in (50, 300, 3000):
    report = estimate_training(repr_name="remi", steps=steps, seq_len=_seq_len, run_benchmark=False)
    est = report["estimate"]
    print(f"steps={steps}: ~{est['estimated_hours']} h ({est['seconds_per_step']} s/step, {est['notes']})")

## Next

1. Run **Sync to Drive** above (if `USE_DRIVE=True`).
2. **Disconnect** this runtime (optional).
3. Open **`02_train_lora_sft.ipynb`** in a **new** session (same `DATASET_MODE`, `RUN_ID`, `USE_DRIVE=True`, `STAGE="S3"` or `"S2"` if you built MidiCaps).